# 00 - Setup: schema, volumes and mock data
Run this notebook **once** before the others. It creates:
* schema `<catalog>.<schema>` (default `main.de_demo`)
* volume `landing` with mock data for Auto Loader: `orders/` (JSON, ~2% dirty rows), `customers_cdc/` (CDC feed: INSERT/UPDATE/DELETE), `events/` (clickstream with late events), `products/` (CSV)
* volume `checkpoints` used by the Structured Streaming notebook

Set the **catalog** and **schema** widgets at the top first. You need `USE CATALOG` + `CREATE SCHEMA` on the catalog (or an existing schema with `CREATE VOLUME` / `CREATE TABLE`). Data is generated with a fixed seed, so re-running produces the same files.

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "de_demo", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
print(f"Target: {CATALOG}.{SCHEMA}  |  landing: {LANDING}")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:s) COMMENT 'Data engineering demo'", args={"s": f"{CATALOG}.{SCHEMA}"})
for vol, comment in [("landing", "Landing zone for Auto Loader ingestion"), ("checkpoints", "Structured Streaming checkpoints")]:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS IDENTIFIER(:v) COMMENT '{comment}'", args={"v": f"{CATALOG}.{SCHEMA}.{vol}"})
display(spark.sql("SHOW VOLUMES IN IDENTIFIER(:s)", args={"s": f"{CATALOG}.{SCHEMA}"}))

In [0]:
import csv, json, os, random
from datetime import datetime, timedelta

random.seed(42)
for d in ["orders", "customers_cdc", "events", "products"]:
    os.makedirs(f"{LANDING}/{d}", exist_ok=True)

def write_jsonl(path, rows):
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

# Products: static CSV dimension
cats = ["Electronics", "Home", "Sports", "Books", "Toys"]
products = [{"product_id": f"P{i:03d}", "product_name": f"Product {i}", "category": cats[i % 5],
             "unit_price": round(random.uniform(5, 500), 2)} for i in range(1, 51)]
with open(f"{LANDING}/products/products.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=products[0].keys())
    w.writeheader()
    w.writerows(products)

# Customers CDC feed: 200 inserts, 40 updates, 10 deletes in 3 files
countries = ["US", "CA", "MX", "BR", "GB", "DE"]
start = datetime(2026, 11, 1)
seq = start
def next_ts():
    global seq
    seq += timedelta(minutes=1)
    return seq.isoformat()

write_jsonl(f"{LANDING}/customers_cdc/customers_cdc_001.json", [
    {"customer_id": i, "name": f"Customer {i}", "email": f"customer{i}@example.com", "country": random.choice(countries),
     "tier": "bronze", "operation": "INSERT", "updated_at": next_ts()} for i in range(1, 201)])
write_jsonl(f"{LANDING}/customers_cdc/customers_cdc_002.json", [
    {"customer_id": i, "name": f"Customer {i}", "email": f"customer{i}@example.com", "country": random.choice(countries),
     "tier": random.choice(["silver", "gold"]), "operation": "UPDATE", "updated_at": next_ts()} for i in random.sample(range(1, 201), 40)])
write_jsonl(f"{LANDING}/customers_cdc/customers_cdc_003.json", [
    {"customer_id": i, "name": None, "email": None, "country": None, "tier": None, "operation": "DELETE",
     "updated_at": next_ts()} for i in random.sample(range(1, 201), 10)])

# Orders: 5 daily files x 1,000 rows, ~1% null customer_id and ~1% negative amounts
order_id = 0
for n in range(1, 6):
    rows = []
    for _ in range(1000):
        order_id += 1
        p = random.choice(products)
        qty = random.randint(1, 5)
        r = {"order_id": order_id, "customer_id": random.randint(1, 200), "product_id": p["product_id"], "quantity": qty,
             "amount": round(qty * p["unit_price"], 2), "status": random.choice(["placed", "shipped", "delivered", "returned"]),
             "order_ts": (start + timedelta(days=n - 1, seconds=random.randint(0, 86399))).isoformat()}
        x = random.random()
        if x < 0.01:
            r["customer_id"] = None
        elif x < 0.02:
            r["amount"] = -r["amount"]
        rows.append(r)
    write_jsonl(f"{LANDING}/orders/orders_{n:03d}.json", rows)

# Clickstream events: 5 hourly files x 2,000 rows, ~3% arrive 10-90 minutes late
for n in range(1, 6):
    t0 = start + timedelta(hours=n)
    rows = []
    for k in range(2000):
        ts = t0 + timedelta(seconds=random.randint(0, 3599))
        if random.random() < 0.03:
            ts -= timedelta(minutes=random.randint(10, 90))
        rows.append({"event_id": f"E{n}-{k}", "user_id": random.randint(1, 200),
                     "event_type": random.choice(["page_view", "add_to_cart", "checkout", "search"]),
                     "page": random.choice(["/home", "/product", "/cart", "/search"]),
                     "device": random.choice(["web", "ios", "android"]), "event_time": ts.isoformat()})
    write_jsonl(f"{LANDING}/events/events_{n:03d}.json", rows)

for d in sorted(os.listdir(LANDING)):
    print(d, sorted(os.listdir(f"{LANDING}/{d}")))

In [0]:
# Expected: orders 5000 (97 invalid), events 10000, customers_cdc 250, products 50
counts = spark.sql(f"""
  SELECT 'orders' AS source, count(*) AS rows, count_if(customer_id IS NULL OR amount < 0) AS invalid_rows
    FROM read_files('{LANDING}/orders', format => 'json')
  UNION ALL SELECT 'events', count(*), NULL FROM read_files('{LANDING}/events', format => 'json')
  UNION ALL SELECT 'customers_cdc', count(*), NULL FROM read_files('{LANDING}/customers_cdc', format => 'json')
  UNION ALL SELECT 'products', count(*), NULL FROM read_files('{LANDING}/products', format => 'csv', header => true)
""")
display(counts)